# 04 - 端到端流程（冒烟 / 全量 time / 全量 sid + 断点续跑）

本 Notebook 是 **`docs/user_guide.md`** 中命令行流水线的交互版，支持 **marker 断点续跑**（日志在 `runs/pipeline-logs/`）。

## 三套流程

| 流程 | marker 前缀 | 实验 YAML | 说明 |
|------|-------------|-----------|------|
| **冒烟** | `smoke` | `exp_baseline.yaml` | ~500 行索引；**关闭 DDP** |
| **全量 time** | `full_best` | `exp_full_best_ddp.yaml` | 全量索引 + **time_stratified** → `data/splits/` |
| **全量 sid** | `full_sid` | `exp_full_best_sid.yaml` | **共用 processed/**；`resplit.sh` → `data/splits_sid/`；跨服务器泛化 |

**推荐顺序**：冒烟 → 全量 time（含 preprocess）→ **全量 sid**（仅 resplit + 训练，不重复扫 mcelog）。

## 产物与指标

- 训练/推理：`runs/exp_tianchi_full_best_ddp/` 与 `runs/exp_tianchi_full_best_sid/`
- **部署 KPI**：`metrics.json` 顶层 / `deployment`（验证集阈值）；**勿**用 `oracle_on_test` 作线上承诺
- 误差分析：`03_error_analysis.ipynb`（FN/FP × `empty_mcelog_window` × sid）

## 使用提示

- marker 前缀 **≠** `run_name`（例：`full_sid` vs `exp_tianchi_full_best_sid`）
- sid 单元要求 `data/processed/preprocess_manifest.json` 存在（先跑全量 time 的 preprocess）
- 改索引/horizon：清空 `data/processed/*` 与各 `splits*` 并删相关 marker

命令行：冒烟 **`bash scripts/autotune.sh`**；全量 **`docs/user_guide.md` §6**；sid **`§6.1`** / **`bash scripts/resplit.sh`**。

In [ ]:
from pathlib import Path
from datetime import datetime
import os

# 从项目根或 notebooks/ 打开均可
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.environ["PYTHONPATH"] = f"{PROJECT_ROOT / 'src'}:{os.environ.get('PYTHONPATH', '')}"

LOG_DIR = PROJECT_ROOT / "runs" / "pipeline-logs"
MARKER_DIR = LOG_DIR / "markers"
LOG_DIR.mkdir(parents=True, exist_ok=True)
MARKER_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOG_DIR:", LOG_DIR)
print("MARKER_DIR:", MARKER_DIR)

In [ ]:
import subprocess

_SRC = PROJECT_ROOT / "src"


def run_step(exp_name: str, step_name: str, command: str) -> None:
    """执行一步 shell 命令；成功则写 marker，便于断点续跑。"""
    marker = MARKER_DIR / f"{exp_name}__{step_name}.done"
    if marker.exists():
        print(f"[SKIP] {exp_name}/{step_name} (marker: {marker.name})")
        return

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = LOG_DIR / f"{exp_name}__{step_name}__{ts}.log"
    cmd = (
        f'bash -lc "set -o pipefail; export PYTHONUNBUFFERED=1; '
        f'export PYTHONPATH={_SRC}:${{PYTHONPATH:-}}; '
        f'{command} 2>&1 | tee \"{log_file}\""'
    )

    print(f"[RUN ] {exp_name}/{step_name}")
    print("      command:", command)
    print("      log:", log_file)
    subprocess.run(cmd, shell=True, check=True, cwd=str(PROJECT_ROOT))
    marker.write_text(ts, encoding="utf-8")
    print(f"[DONE] {exp_name}/{step_name}\n")


def clear_exp_markers(exp_name: str) -> int:
    """删除某一实验（如 smoke / full_best）的全部 marker，便于该流程从头重跑。"""
    removed = 0
    for p in sorted(MARKER_DIR.glob(f"{exp_name}__*.done")):
        p.unlink(missing_ok=True)
        removed += 1
    print(f"已删除 {exp_name} 的 marker 数量: {removed}")
    return removed

In [ ]:
# 通用预检：切分形状与标签（建议在 train 前运行；传入对应实验 YAML）
import numpy as np
import sys

_src = str(PROJECT_ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from dram2cnn.utils.config import load_experiment_config


def _splits_dir_for_exp(exp_yaml: str, root: Path | None = None) -> Path:
    root = root or PROJECT_ROOT
    cfg = load_experiment_config(root / exp_yaml)
    p = Path(cfg["data"]["splits_dir"])
    return p if p.is_absolute() else root / p


def preflight_splits_vs_model(exp_yaml: str, root: Path | None = None) -> None:
    """比对 X_train 与**当前实验合并后**的 model 配置（smoke→model_smoke，全量→model.yaml）。"""
    root = root or PROJECT_ROOT
    spl = _splits_dir_for_exp(exp_yaml, root)
    x_train = spl / "X_train.npy"
    if not x_train.is_file():
        print(f"[preflight] 尚无 {x_train}，请先 preprocess / resplit")
        return
    cfg = load_experiment_config(root / exp_yaml)
    model_cfg = cfg["model"]
    exp_c = int(model_cfg["in_channels"])
    exp_h = int(model_cfg["grid_height"])
    exp_w = int(model_cfg["grid_width"])
    x = np.load(x_train, mmap_mode="r")
    _n, c, h, w = (int(x.shape[i]) for i in range(4))
    if (c, h, w) == (exp_c, exp_h, exp_w):
        print(f"[preflight] 通过 ({exp_yaml}): X_train (C,H,W)=({c},{h},{w}) splits={spl}")
    else:
        print(
            f"[preflight] 不匹配 ({exp_yaml}): X_train=({c},{h},{w}) "
            f"vs model=({exp_c},{exp_h},{exp_w})；删对应 preprocess/resplit marker 后重跑"
        )


def preflight_split_labels(exp_yaml: str, root: Path | None = None) -> None:
    root = root or PROJECT_ROOT
    spl = _splits_dir_for_exp(exp_yaml, root)
    man = spl / "split_manifest.json"
    if man.is_file():
        import json
        print(f"[preflight] split_manifest ({man.name}):", json.loads(man.read_text())["split_strategy"])
    for tag in ("train", "val", "test"):
        p = spl / f"y_{tag}.npy"
        if not p.is_file():
            continue
        y = np.load(p)
        pos, n = int(y.sum()), int(len(y))
        if pos == 0:
            print(f"[preflight] 警告 {exp_yaml} {tag}: 无正样本 → AUC/PR 可能为 nan")
        else:
            print(f"[preflight] {exp_yaml} {tag}: N={n} pos={pos} rate={pos / max(n, 1):.4f}")

---
## 冒烟测试（Smoke）

目标：快速验证 **索引 → 预处理 → 训练 → 推理 → 评估** 全链路可用。

| 步骤 | 说明 |
|------|------|
| 1. 生成演示索引 | `features_5m_index_demo.csv`（~500 行） |
| 2. preprocess | `exp_baseline.yaml`，截断 mcelog/索引 |
| 3. train | `train.sh`（exp_baseline **关闭 DDP** 以加速） |
| 4. infer / evaluate | 推理与指标 |

命令行一键等价（无 marker）：**`bash scripts/autotune.sh`** = preprocess（含 validate）→ train → infer → evaluate（**不含** build_index/inspect_data；完整冒烟见本 Notebook 冒烟单元）。

In [ ]:
# 【冒烟 1/2】重建演示索引（与 configs/data.yaml 默认 index_features_path 一致）
SMOKE_EXP = "smoke"
run_step(SMOKE_EXP, "build_index", "bash scripts/build_index.sh smoke")

In [ ]:
# 【冒烟 2/2】baseline 全流程（configs/experiments/exp_baseline.yaml）
SMOKE_CFG = "configs/experiments/exp_baseline.yaml"

run_step(SMOKE_EXP, "preprocess", f"bash scripts/preprocess.sh {SMOKE_CFG}")
# preprocess.sh 末尾已 validate；以下为可选重复诊断
# run_step(SMOKE_EXP, "validate", f"bash scripts/validate.sh {SMOKE_CFG}")
run_step(SMOKE_EXP, "inspect_data", f"bash scripts/inspect_data.sh {SMOKE_CFG}")

preflight_splits_vs_model(SMOKE_CFG, PROJECT_ROOT)
preflight_split_labels(SMOKE_CFG, PROJECT_ROOT)

run_step(SMOKE_EXP, "train", f"bash scripts/train.sh {SMOKE_CFG}")
run_step(SMOKE_EXP, "infer", f"bash scripts/infer.sh {SMOKE_CFG}")
run_step(SMOKE_EXP, "evaluate", f"bash scripts/evaluate.sh {SMOKE_CFG}")

print("冒烟完成 → runs/exp_tianchi_baseline/")
print("后续分析：02_feature_analysis.ipynb（X 分布）、03_error_analysis.ipynb（FP/FN，EXP 仍用 exp_baseline）")

In [ ]:
# 【可选】命令行 autotune 一键冒烟（不分 marker；与上一单元二选一即可）
# run_step(SMOKE_EXP, "autotune", "bash scripts/autotune.sh")

---
## 全量测试（Full + DDP）

目标：在 **真实全量索引 + 完整 mcelog** 上训练（多卡 DDP）。

| 步骤 | 说明 |
|------|------|
| 1. 生成全量索引 | 输出 `data/raw/features_index_h300s_step300s_full.csv`（耗时长） |
| 2. **配置** | `exp_full_best_ddp.yaml` 已含全量索引；**`focal_gamma: 0`**（Deep 模型 + Focal 易 NaN）；系统 **RAM 峰值 ≤50%** |
| 3. 清空旧产物 | 若曾跑冒烟 preprocess：`rm -rf data/processed/* data/splits/*` 及 `full_best__preprocess.done` |
| 4. preprocess | **建议在终端**跑；日志应含「目标≤50%」、worker 通常=1 |
| 5. train_ddp / infer / evaluate | preprocess 完成后 **Kernel Restart**，再跑后续单元 |

> **注意**：仅取消 `max_index_rows` 而不改 `index_features_path` 时，仍会用 ~500 行演示索引。  
> 终端等价：`bash scripts/preprocess.sh configs/experiments/exp_full_best_ddp.yaml`

In [ ]:
# 【全量 1/3】扫描完整 mcelog 生成全量索引（耗时长，可跳过若 CSV 已存在且 marker 在）
FULL_EXP = "full_best"
run_step(FULL_EXP, "build_index", "bash scripts/build_index.sh full")

print(
    "\n下一步：\n"
    "  1) exp_full_best_ddp.yaml 已默认 index_features_path=data/raw/features_index_h300s_step300s_full.csv\n"
    "  2) 若曾跑过冒烟 preprocess：rm -rf data/processed/* data/splits/*\n"
    "  3) clear_exp_markers('full_best')  # 或删 full_best__preprocess.done\n"
    "  4) 运行下方「全量预检 + 流水线」单元（preprocess 建议终端执行）"
)

In [ ]:
# 【全量 2/3】预检：index_features_path 不得仍指向演示 stub
import sys

_src = str(PROJECT_ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from dram2cnn.utils.config import load_experiment_config
from dram2cnn.utils.experiment_guards import assert_full_best_index_not_stub

FULL_CFG = "configs/experiments/exp_full_best_ddp.yaml"
_cfg = load_experiment_config(PROJECT_ROOT / FULL_CFG)
assert_full_best_index_not_stub(_cfg, context="04_full_best_preflight")
print(f"[preflight] 全量 index_features_path OK: {_cfg['data']['index_features_path']}")

In [ ]:
# 【全量 3/3】exp_full_best_ddp：preprocess → DDP 训练 → infer → evaluate
run_step(FULL_EXP, "preprocess", f"bash scripts/preprocess.sh {FULL_CFG}")
# run_step(FULL_EXP, "validate", f"bash scripts/validate.sh {FULL_CFG}")
run_step(FULL_EXP, "inspect_data", f"bash scripts/inspect_data.sh {FULL_CFG}")

preflight_splits_vs_model(FULL_CFG, PROJECT_ROOT)
preflight_split_labels(FULL_CFG, PROJECT_ROOT)

# 多卡：运行前在终端 export CUDA_VISIBLE_DEVICES=0,1,2
run_step(FULL_EXP, "train", f"bash scripts/train_ddp.sh {FULL_CFG}")
run_step(FULL_EXP, "infer", f"bash scripts/infer.sh {FULL_CFG}")
run_step(FULL_EXP, "evaluate", f"bash scripts/evaluate.sh {FULL_CFG}")

print("全量完成 → runs/exp_tianchi_full_best_ddp/")
print("分析 03 时请将 EXP_YAML 改为 exp_full_best_ddp.yaml，或手动指定 predictions 路径")

---
## 全量 sid 泛化对比（Full + sid split）

目标：在**同一** `data/processed/` 上按 **sid** 切分（同一台服务器仅落一个 split），与 **time_stratified**（`full_best`）对比 test 指标。

| 步骤 | 说明 |
|------|------|
| 0. 前置 | 已完成 **全量 time** 的 `preprocess`（`data/processed/` 就绪） |
| 1. resplit | `bash scripts/resplit.sh` → `data/splits_sid/split_manifest.json` |
| 2. train_ddp / infer / evaluate | 产物 `runs/exp_tianchi_full_best_sid/` |
| 3. 对比 | 下一单元格读取 time vs sid 的 **deployment F1** |

命令行等价：**`docs/user_guide.md` §6.1**

In [ ]:
# 【sid 1/2】预检 processed + resplit → train → infer → evaluate
SID_EXP = "full_sid"
SID_CFG = "configs/experiments/exp_full_best_sid.yaml"

_proc_man = PROJECT_ROOT / "data/processed/preprocess_manifest.json"
if not _proc_man.is_file():
    raise RuntimeError(
        "缺少 data/processed/preprocess_manifest.json。\n"
        "请先运行上方「全量 time」的 preprocess 单元，或：\n"
        "  bash scripts/preprocess.sh configs/experiments/exp_full_best_ddp.yaml"
    )
print("[preflight] processed OK:", _proc_man)

run_step(SID_EXP, "resplit", f"bash scripts/resplit.sh {SID_CFG}")
run_step(SID_EXP, "inspect_data", f"bash scripts/inspect_data.sh {SID_CFG}")

preflight_splits_vs_model(SID_CFG, PROJECT_ROOT)
preflight_split_labels(SID_CFG, PROJECT_ROOT)

run_step(SID_EXP, "train", f"bash scripts/train_ddp.sh {SID_CFG}")
run_step(SID_EXP, "infer", f"bash scripts/infer.sh {SID_CFG}")
run_step(SID_EXP, "evaluate", f"bash scripts/evaluate.sh {SID_CFG}")

print("sid 完成 → runs/exp_tianchi_full_best_sid/")
print("分析：03_error_analysis.ipynb，EXP_YAML=exp_full_best_sid.yaml")

In [ ]:
# 【sid 2/2】time_stratified vs sid：部署 KPI 对比（deployment F1，非 oracle）
import json

from dram2cnn.utils.run_layout import read_pointer_dir


def _load_deploy_metrics(run_name: str) -> dict | None:
    root = PROJECT_ROOT / "runs" / run_name
    infer_dir = read_pointer_dir(root, ".latest_infer_session", "latest_infer")
    if infer_dir is None:
        return None
    mp = infer_dir / "metrics.json"
    if not mp.is_file():
        return None
    payload = json.loads(mp.read_text(encoding="utf-8"))
    dep = payload.get("deployment") or {}
    m = dep.get("metrics") or payload
    return {
        "run_name": run_name,
        "threshold": dep.get("threshold"),
        "f1": m.get("f1"),
        "precision": m.get("precision"),
        "recall": m.get("recall"),
        "roc_auc": m.get("roc_auc"),
        "metrics_path": str(mp),
    }


rows = []
for rn, split_label in (
    ("exp_tianchi_full_best_ddp", "time_stratified"),
    ("exp_tianchi_full_best_sid", "sid"),
):
    row = _load_deploy_metrics(rn)
    if row:
        row["split"] = split_label
        rows.append(row)
    else:
        print(f"提示：尚无 {rn} 的 metrics.json，请先跑对应 train/infer/evaluate")

if rows:
    import pandas as pd

    cmp_df = pd.DataFrame(rows)[
        ["split", "f1", "precision", "recall", "roc_auc", "threshold", "metrics_path"]
    ]
    print(cmp_df.to_string(index=False))
    out_cmp = PROJECT_ROOT / "outputs" / "tables"
    out_cmp.mkdir(parents=True, exist_ok=True)
    cmp_df.to_csv(out_cmp / "full_best_time_vs_sid_deployment_kpi.csv", index=False)
    print("\n已写入 outputs/tables/full_best_time_vs_sid_deployment_kpi.csv")
    print("说明：仅 deployment 口径；oracle_on_test 见各 metrics.json 内 oracle_on_test 段。")

In [ ]:
# 查看最近日志与 marker
print("\n=== 最近日志（最多 15 条）===")
for p in sorted(LOG_DIR.glob("*.log"), key=lambda x: x.stat().st_mtime, reverse=True)[:15]:
    print(p.name)

print("\n=== 已完成 marker ===")
for p in sorted(MARKER_DIR.glob("*.done")):
    print(p.name, "=>", p.read_text(encoding="utf-8").strip())

In [ ]:
# 重置 marker（可选 — 取消注释后再运行本格）
# clear_exp_markers("smoke")      # 仅重跑冒烟
# clear_exp_markers("full_best")  # 仅重跑全量 time
# clear_exp_markers("full_sid")   # 仅重跑 sid 对比
#
# CLEAR_ALL_MARKERS = False
# if CLEAR_ALL_MARKERS:
#     removed = 0
#     for p in sorted(MARKER_DIR.glob("*.done")):
#         p.unlink(missing_ok=True)
#         removed += 1
#     print(f"已删除全部 marker 数量: {removed}")
# else:
print("未删除 marker。需重跑某流程时取消上方 clear_exp_markers(...) 注释。")